## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-17-02-46-00 +0000,wsj,Singapore Dollar Weakens Slightly on Risk-Off ...,https://www.wsj.com/finance/currencies/asian-c...
1,2026-11-17-02-46-00 +0000,wsj,Trump’s Advisers Are at War Over China Ahead o...,https://www.wsj.com/world/china/trumps-adviser...
2,2026-11-17-02-40-29 +0000,48hills,Protest calls for AI treaty,https://48hills.org/2026/11/protest-calls-for-...
3,2026-11-17-02-39-08 +0000,startribune,Live: Wolves lead Trail Blazers by double digi...,https://www.startribune.com/minnesota-timberwo...
4,2026-11-17-02-38-00 +0000,wsj,Nvidia Board Warned of ‘Irreplaceable’ Talent ...,https://www.wsj.com/business/nvidia-board-warn...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2530/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
7,trump,42
27,nvidia,26
10,china,19
16,ai,17
446,new,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
216,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...,125
229,2026-11-16-17-02-28 +0000,chitri,President Trump backs campaign to reinstate Nv...,https://www.chicagotribune.com/2026/11/16/nvid...,118
277,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...,115
5,2026-11-17-02-37-53 +0000,bbc,Nvidia dismisses 'Bring Jensen Back' campaign ...,https://www.bbc.co.uk/news/articles/cry49plv8v...,109
206,2026-11-16-17-48-36 +0000,nyt,Trump Says He’s Open to Reinstatement of Jense...,https://www.nytimes.com/2026/11/16/us/politics...,108


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
216,125,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...
227,79,2026-11-16-17-04-16 +0000,nypost,Commerce Department moves to crack down on Chi...,https://nypost.com/2026/11/16/business/commerc...
225,51,2026-11-16-17-08-46 +0000,wapo,New trial is set in Lindsay Clancy case over t...,https://www.washingtonpost.com/nation/2026/11/...
26,47,2026-11-17-01-00-00 +0000,wsj,"Fresh off big state-level victories, Democrats...",https://www.wsj.com/politics/policy/in-blue-wa...
103,46,2026-11-16-22-32-19 +0000,startribune,Attack leaves Somali American Uber driver in c...,https://www.startribune.com/somali-uber-driver...
56,43,2026-11-16-23-52-35 +0000,nypost,Newsom’s death row ban hits major snag as vict...,https://nypost.com/2026/11/16/us-news/families...
291,43,2026-11-16-13-43-51 +0000,cbc,Bank of Canada likely to hold interest rates s...,https://www.cbc.ca/news/business/cpi-inflation...
145,43,2026-11-16-21-06-54 +0000,nypost,JD Vance promises Iran won’t turn into ‘foreve...,https://nypost.com/2026/11/16/us-news/jd-vance...
5,41,2026-11-17-02-37-53 +0000,bbc,Nvidia dismisses 'Bring Jensen Back' campaign ...,https://www.bbc.co.uk/news/articles/cry49plv8v...
378,40,2026-11-16-05-54-19 +0000,latimes,Family of Franklin High student who died at sc...,https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
